# requires-grad-propagation composite — cx11: Recipe presence iff requires_grad propagates through composed forward calls

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `recipe-dataclass`, `requires-grad-propagation`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "requires-grad-propagation"
DD_ATOM_IDS = ["recipe-dataclass", "requires-grad-propagation"]
DD_SUBTOPICS = ["Backprop: Recipe dataclass", "Backprop: requires_grad propagation"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Recipe presence ⟺ requires_grad — they MUST agree

An invariant the reverse pass relies on:

- A MiniTensor with `requires_grad=True` MUST carry a Recipe (unless it's a leaf — leaves have `recipe is None` AND set rg=True directly).
- A MiniTensor with `requires_grad=False` MUST have `recipe is None`.

When ops compose, this invariant must hold through every link:
`f(g(x))` propagates rg=True ONLY if g produced an output with rg=True, which only happens if any of g's inputs had rg=True. The Recipe-chain and the rg-chain are the same chain — viewed two different ways.

### Composite Exercise — Recipe presence iff requires_grad propagates through composed forward calls

**Atoms exercised together**: `recipe-dataclass`, `requires-grad-propagation`

Implement two helpers and verify they compose correctly:

**(1)** `cx11_forward(fwd_fn, args, is_differentiable=True)` — a single forward call that produces a `MiniTensor` with:
- `requires_grad = is_differentiable AND any(isinstance(a, MiniTensor) and a.requires_grad for a in args)` (constants don't block, `is_differentiable` can veto).
- A Recipe attached IFF `requires_grad` is True. The Recipe holds the raw args (after unbox), empty kwargs, and a parents-by-argidx dict.

**(2)** `cx11_chain(x, *fwd_fns)` — chain forward calls `f1(f2(...(fn(x))))`. The final output's `requires_grad` must equal `x.requires_grad`, and the chain of Recipes must be intact (every non-leaf in the chain has a Recipe pointing to the previous link as parent at argidx 0).

In [ ]:
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe

def cx11_forward(fwd_fn, args, is_differentiable=True):
    raw_args = tuple(
        a.array if isinstance(a, MiniTensor) else a
        for a in args
    )
    parents = {
        idx: a
        for idx, a in enumerate(args)
        if isinstance(a, MiniTensor)
    }
    any_tracked = any(
        isinstance(a, MiniTensor) and a.requires_grad for a in args
    )
    requires_grad = is_differentiable and any_tracked
    raw_out = fwd_fn(*raw_args)
    out = MiniTensor(raw_out, requires_grad=requires_grad)
    if requires_grad:
        out.recipe = Recipe(fwd_fn, raw_args, {}, parents)
    return out

def cx11_chain(x, *fwd_fns):
    cur = x
    for f in fwd_fns:
        cur = cx11_forward(f, (cur,))
    return cur


<details><summary>Show solution — cx11</summary>

```python
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe

def cx11_forward(fwd_fn, args, is_differentiable=True):
    raw_args = tuple(
        a.array if isinstance(a, MiniTensor) else a
        for a in args
    )
    parents = {
        idx: a
        for idx, a in enumerate(args)
        if isinstance(a, MiniTensor)
    }
    any_tracked = any(
        isinstance(a, MiniTensor) and a.requires_grad for a in args
    )
    requires_grad = is_differentiable and any_tracked
    raw_out = fwd_fn(*raw_args)
    out = MiniTensor(raw_out, requires_grad=requires_grad)
    if requires_grad:
        out.recipe = Recipe(fwd_fn, raw_args, {}, parents)
    return out

def cx11_chain(x, *fwd_fns):
    cur = x
    for f in fwd_fns:
        cur = cx11_forward(f, (cur,))
    return cur
```

**The biconditional `rg ⟺ recipe-present` (non-leaf).** This drill tests both directions:
- rg=True ⇒ Recipe attached (so reverse pass has somewhere to start).
- rg=False ⇒ Recipe is None (so inference doesn't accumulate state).

Leaves are the carve-out: `x = MiniTensor(arr, requires_grad=True)` has `rg=True` AND `recipe is None`. The reverse pass treats `recipe is None` as the stop signal — it then writes the accumulated grad into `x.grad`.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx11'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx11',
        'subtopics': ["Backprop: Recipe dataclass", "Backprop: requires_grad propagation"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()